# Coastal and river flood avoided EAD and NbS benefit locations

Separate comparison panel combining the coastal-flood maps from the paper 3 analysis with the published river-flood forest-restoration maps from Haggis et al. This notebook leaves the coastal-only figure unchanged and writes new comparison outputs.

In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator, NullLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.plot import plotting_extent


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "dphil_papers").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing dphil_papers")


ROOT = find_project_root()
PAPERS = ROOT / "dphil_papers"
ROBYN_LIBRARIES = PAPERS / "robyns_libraries"
if str(ROBYN_LIBRARIES) not in sys.path:
    sys.path.append(str(ROBYN_LIBRARIES))

import Robyn_paper_2_defs

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [ ]:
PAPER2 = PAPERS / "dphil_paper_2"
PAPER3 = PAPERS / "dphil_paper_3"
COMMON = PAPERS / "dphil_common_cross_cutting"

CRS_METRIC = "EPSG:3448"
JMD_TO_USD = 1.0 / 150.0
COASTAL_ASSET_COLOR_CAP_QUANTILE = 0.995
COASTAL_MANGROVE_COLOR_CAP_QUANTILE = 0.995
RIVER_RESTORATION_COLOR_CAP_QUANTILE = 0.995
LANDSLIDE_PATCH_COLOR_CAP_QUANTILE = 0.98
RIVER_AVOIDED_EAD_COLOR_CAP_USD = 25_000

BOUNDARY_PATH = COMMON / "common_incoming_data" / "boundaries" / "jam_adm_shp" / "jam_admbnda_adm0.shp"
COASTAL_ASSET_LOCATIONS_PATH = PAPER3 / "results" / "03_cross_hazard_comparison" / "spatial_service_provision_comparison" / "coastal_avoided_ead_mangrove_locations" / "coastal_positive_avoided_asset_locations_maximum.geoparquet"
COASTAL_MANGROVE_ATTRIBUTION_PATH = PAPER3 / "results_coastal_scenario_comparison" / "weighted_area_distance_signed" / "mangrove_priority_ranking" / "mangrove_priority_map_weighted_area_distance.gpkg"
RIVER_AVOIDED_EAD_RASTER_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "avoided_fluvial_eads" / "avoided__fluvial__ead_max_300m_smoothed.tif"
RIVER_RESTORATION_BENEFIT_RASTER_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_max.tif"
LANDSLIDE_ASSET_MAP_LAYERS_PATH = PAPER3 / "results" / "02_damage_estimates" / "landslide_damages" / "results_landslide_maximum_scenario_combined_class" / "damage_estimates" / "landslide_source_and_runout_ead_asset_map_layers_combined_class.gpkg"
LANDSLIDE_PATCH_ATTRIBUTION_DIR = PAPER3 / "results" / "02_damage_estimates" / "landslide_damages" / "source_id_attribution_combined_class" / "maximum" / "forest_patch_attribution"
LANDSLIDE_PROTECTION_PATCH_CELL_RASTER_PATH = LANDSLIDE_PATCH_ATTRIBUTION_DIR / "landslide_protection_existing_forest_cell_attribution_usd_combined_class.tif"
LANDSLIDE_RESTORATION_PATCH_CELL_RASTER_PATH = LANDSLIDE_PATCH_ATTRIBUTION_DIR / "landslide_restoration_afforestable_cell_attribution_usd_combined_class.tif"

OUTPUT_DIR = PAPER3 / "results" / "03_cross_hazard_comparison" / "spatial_service_provision_comparison" / "coastal_river_location_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for input_path in [
    BOUNDARY_PATH,
    COASTAL_ASSET_LOCATIONS_PATH,
    COASTAL_MANGROVE_ATTRIBUTION_PATH,
    RIVER_AVOIDED_EAD_RASTER_PATH,
    RIVER_RESTORATION_BENEFIT_RASTER_PATH,
    LANDSLIDE_ASSET_MAP_LAYERS_PATH,
    LANDSLIDE_PROTECTION_PATCH_CELL_RASTER_PATH,
    LANDSLIDE_RESTORATION_PATCH_CELL_RASTER_PATH,
]:
    if not input_path.exists():
        raise FileNotFoundError(input_path)

OUTPUT_DIR


## Load map layers

In [ ]:
boundary = gpd.read_file(BOUNDARY_PATH).to_crs(CRS_METRIC)
boundary_parts = boundary.explode(index_parts=False).copy()
main_island_boundary = boundary_parts.loc[boundary_parts.geometry.area > 1_000_000].copy()
MAIN_ISLAND_BOUNDS = main_island_boundary.total_bounds

positive_coastal_asset_locations = gpd.read_parquet(COASTAL_ASSET_LOCATIONS_PATH).to_crs(CRS_METRIC)
coastal_mangrove_attribution = gpd.read_file(COASTAL_MANGROVE_ATTRIBUTION_PATH).to_crs(CRS_METRIC)
positive_coastal_mangrove_attribution = coastal_mangrove_attribution.loc[
    coastal_mangrove_attribution["avoided_usd_max"] > 0
].copy()

positive_coastal_asset_locations["plot_avoided_ead_usd"] = positive_coastal_asset_locations[
    "Avoided_EAD_USD"
].clip(
    upper=positive_coastal_asset_locations["Avoided_EAD_USD"].quantile(COASTAL_ASSET_COLOR_CAP_QUANTILE)
)
positive_coastal_mangrove_attribution["plot_avoided_ead_thousand_usd"] = (
    positive_coastal_mangrove_attribution["avoided_usd_max"] / 1_000
).clip(
    upper=(positive_coastal_mangrove_attribution["avoided_usd_max"] / 1_000).quantile(
        COASTAL_MANGROVE_COLOR_CAP_QUANTILE
    )
)

landslide_asset_locations = gpd.read_file(LANDSLIDE_ASSET_MAP_LAYERS_PATH).to_crs(CRS_METRIC)
positive_landslide_protection_locations = landslide_asset_locations.loc[
    landslide_asset_locations["Avoided_EAD_Protection_USD"] > 0
].copy()
positive_landslide_restoration_locations = landslide_asset_locations.loc[
    landslide_asset_locations["Avoided_EAD_Reafforestation_USD"] > 0
].copy()
positive_landslide_protection_locations["plot_avoided_ead_usd"] = positive_landslide_protection_locations[
    "Avoided_EAD_Protection_USD"
].clip(
    upper=positive_landslide_protection_locations["Avoided_EAD_Protection_USD"].quantile(0.995)
)
positive_landslide_restoration_locations["plot_avoided_ead_usd"] = positive_landslide_restoration_locations[
    "Avoided_EAD_Reafforestation_USD"
].clip(
    upper=positive_landslide_restoration_locations["Avoided_EAD_Reafforestation_USD"].quantile(0.995)
)

def read_raster_as_usd(
    raster_path: Path,
    minimum_positive_usd: float = 0.0,
    value_multiplier: float = JMD_TO_USD,
) -> dict:
    with rasterio.open(raster_path) as raster_dataset:
        raster_values = raster_dataset.read(1, masked=True)
        raster_extent = plotting_extent(raster_dataset)
        raster_crs = raster_dataset.crs

    data_usd = np.array(raster_values, dtype="float64") * value_multiplier
    data_usd[raster_values.mask | (data_usd <= minimum_positive_usd)] = np.nan
    return {"data": data_usd, "extent": raster_extent, "crs": raster_crs}


river_avoided_ead = read_raster_as_usd(
    RIVER_AVOIDED_EAD_RASTER_PATH,
    minimum_positive_usd=JMD_TO_USD,
)
river_restoration_benefit = read_raster_as_usd(
    RIVER_RESTORATION_BENEFIT_RASTER_PATH,
    minimum_positive_usd=0.0,
)
landslide_protection_patch_benefit = read_raster_as_usd(
    LANDSLIDE_PROTECTION_PATCH_CELL_RASTER_PATH,
    minimum_positive_usd=0.0,
    value_multiplier=1.0,
)
landslide_restoration_patch_benefit = read_raster_as_usd(
    LANDSLIDE_RESTORATION_PATCH_CELL_RASTER_PATH,
    minimum_positive_usd=0.0,
    value_multiplier=1.0,
)

river_restoration_positive_values = river_restoration_benefit["data"][
    np.isfinite(river_restoration_benefit["data"])
]
RIVER_RESTORATION_COLOR_CAP_USD = float(
    np.nanquantile(river_restoration_positive_values, RIVER_RESTORATION_COLOR_CAP_QUANTILE)
)

landslide_protection_patch_positive_values = landslide_protection_patch_benefit["data"][
    np.isfinite(landslide_protection_patch_benefit["data"])
]
landslide_restoration_patch_positive_values = landslide_restoration_patch_benefit["data"][
    np.isfinite(landslide_restoration_patch_benefit["data"])
]
LANDSLIDE_PROTECTION_PATCH_COLOR_CAP_USD = float(
    np.nanquantile(landslide_protection_patch_positive_values, LANDSLIDE_PATCH_COLOR_CAP_QUANTILE)
)
LANDSLIDE_RESTORATION_PATCH_COLOR_CAP_USD = float(
    np.nanquantile(landslide_restoration_patch_positive_values, LANDSLIDE_PATCH_COLOR_CAP_QUANTILE)
)

summary = pd.DataFrame(
    [
        {
            "layer": "coastal_positive_asset_locations",
            "feature_count": len(positive_coastal_asset_locations),
            "total_positive_avoided_ead_usd": positive_coastal_asset_locations["Avoided_EAD_USD"].sum(),
            "colour_cap_usd": positive_coastal_asset_locations["plot_avoided_ead_usd"].max(),
        },
        {
            "layer": "coastal_positive_mangrove_patches",
            "feature_count": len(positive_coastal_mangrove_attribution),
            "total_positive_avoided_ead_usd": positive_coastal_mangrove_attribution["avoided_usd_max"].sum(),
            "colour_cap_usd": positive_coastal_mangrove_attribution["plot_avoided_ead_thousand_usd"].max() * 1_000,
        },
        {
            "layer": "river_avoided_ead_raster",
            "feature_count": int(np.isfinite(river_avoided_ead["data"]).sum()),
            "total_positive_avoided_ead_usd": np.nansum(river_avoided_ead["data"]),
            "colour_cap_usd": RIVER_AVOIDED_EAD_COLOR_CAP_USD,
        },
        {
            "layer": "river_restoration_benefit_raster",
            "feature_count": int(np.isfinite(river_restoration_benefit["data"]).sum()),
            "total_positive_avoided_ead_usd": np.nansum(river_restoration_benefit["data"]),
            "colour_cap_usd": RIVER_RESTORATION_COLOR_CAP_USD,
        },
        {
            "layer": "landslide_protection_asset_locations",
            "feature_count": len(positive_landslide_protection_locations),
            "total_positive_avoided_ead_usd": positive_landslide_protection_locations["Avoided_EAD_Protection_USD"].sum(),
            "colour_cap_usd": positive_landslide_protection_locations["plot_avoided_ead_usd"].max(),
        },
        {
            "layer": "landslide_restoration_asset_locations",
            "feature_count": len(positive_landslide_restoration_locations),
            "total_positive_avoided_ead_usd": positive_landslide_restoration_locations["Avoided_EAD_Reafforestation_USD"].sum(),
            "colour_cap_usd": positive_landslide_restoration_locations["plot_avoided_ead_usd"].max(),
        },
        {
            "layer": "landslide_protection_existing_forest_cell_attribution",
            "feature_count": int(np.isfinite(landslide_protection_patch_benefit["data"]).sum()),
            "total_positive_avoided_ead_usd": np.nansum(landslide_protection_patch_benefit["data"]),
            "colour_cap_usd": LANDSLIDE_PROTECTION_PATCH_COLOR_CAP_USD,
        },
        {
            "layer": "landslide_restoration_afforestable_cell_attribution",
            "feature_count": int(np.isfinite(landslide_restoration_patch_benefit["data"]).sum()),
            "total_positive_avoided_ead_usd": np.nansum(landslide_restoration_patch_benefit["data"]),
            "colour_cap_usd": LANDSLIDE_RESTORATION_PATCH_COLOR_CAP_USD,
        },
    ]
)
summary.to_csv(OUTPUT_DIR / "coastal_river_location_comparison_summary.csv", index=False)
summary


## Plotting helpers

In [ ]:
NATURE_RC = {
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "figure.titlesize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}

ASSET_CMAP = mpl.colormaps["magma_r"].copy()
ASSET_CMAP.set_bad((0, 0, 0, 0))
MANGROVE_CMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "mangrove_green_blue_black",
    ["#66bd63", "#1a9850", "#2b8cbe", "#08519c", "#000000"],
)
RIVER_RESTORATION_CMAP = mpl.colormaps["Greens"].copy()
RIVER_RESTORATION_CMAP.set_bad((0, 0, 0, 0))
LANDSLIDE_PATCH_CMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "landslide_patch_green_black",
    ["#edf8e9", "#bae4b3", "#41ab5d", "#006d2c", "#000000"],
)
LANDSLIDE_PATCH_CMAP.set_bad((0, 0, 0, 0))
BOUNDARY_COLOR = "#4d4d4d"
NO_BENEFIT_MANGROVE_COLOR = "#f1f1f1"
POSITIVE_MANGROVE_EDGE_COLOR = "#005a32"


def format_number_tick(value: float) -> str:
    if value >= 1_000:
        return f"{value:,.0f}"
    if value >= 10:
        return f"{value:.0f}"
    return f"{value:.1f}"


def set_map_extent(axis, map_bounds, right_padding_fraction: float = 0.08) -> None:
    x_min, y_min, x_max, y_max = map_bounds
    x_range = x_max - x_min
    y_range = y_max - y_min
    axis.set_xlim(x_min - 0.02 * x_range, x_max + right_padding_fraction * x_range)
    axis.set_ylim(y_min - 0.05 * y_range, y_max + 0.08 * y_range)


def add_standard_map_furniture(axis, compact: bool = False) -> None:
    scale_bar_point = Robyn_paper_2_defs.add_scale_bar(
        axis,
        main_island_boundary,
        where="right-top",
        pad=0.07,
        length_km=20,
        max_frac=0.22,
        lw=0.45 if compact else 0.5,
        tick_h_frac=0.010 if compact else 0.012,
        fs_lab=5 if compact else 5.5,
        fs_unit=5 if compact else 5.5,
        unit_text="km",
    )
    if scale_bar_point is None:
        return

    center_data_x, center_data_y = scale_bar_point
    center_axes_x, center_axes_y = axis.transAxes.inverted().transform(
        axis.transData.transform((center_data_x, center_data_y))
    )
    Robyn_paper_2_defs.add_north_arrow_axes(
        axis,
        center_axes_x,
        center_axes_y,
        size_frac=0.055 if compact else 0.065,
        gap_frac=0.030 if compact else 0.035,
        shaft_w_frac=0.10,
        head_w_frac=0.32,
        head_h_frac=0.55,
        fs=5.5 if compact else 6,
        lw=0.45 if compact else 0.5,
    )


def add_colorbar(figure, axis, cmap, norm: Normalize, label: str, top_label_prefix: str | None = None) -> None:
    divider = make_axes_locatable(axis)
    colorbar_axis = divider.append_axes("right", size="2.6%", pad=0.015)
    scalar_mappable = ScalarMappable(norm=norm, cmap=cmap)
    scalar_mappable.set_array([])
    colorbar = figure.colorbar(scalar_mappable, cax=colorbar_axis)
    colorbar.ax.yaxis.set_minor_locator(NullLocator())
    colorbar.outline.set_linewidth(0.35)
    colorbar.ax.tick_params(width=0.35, length=2, labelsize=6)
    colorbar.set_label(label, fontsize=6.5)

    locator = MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10], min_n_ticks=4)
    tick_values = locator.tick_values(norm.vmin, norm.vmax)
    tick_values = tick_values[(tick_values >= norm.vmin) & (tick_values <= norm.vmax)]
    if len(tick_values) == 0:
        tick_values = np.array([norm.vmin, norm.vmax])
    elif abs(tick_values[-1] - norm.vmax) <= 0.08 * max(abs(norm.vmax), 1.0):
        tick_values[-1] = norm.vmax
    else:
        tick_values = np.append(tick_values, norm.vmax)

    tick_labels = [format_number_tick(tick_value) for tick_value in tick_values]
    if top_label_prefix is not None and tick_labels:
        tick_labels[-1] = f"{top_label_prefix}{tick_labels[-1]}"
    colorbar.set_ticks(tick_values)
    colorbar.set_ticklabels(tick_labels)


def add_panel_label(axis, label: str) -> None:
    axis.text(
        0.01,
        0.98,
        label,
        transform=axis.transAxes,
        ha="left",
        va="top",
        fontsize=9,
        fontweight="bold",
    )


def plot_geometry_types(axis, geodataframe: gpd.GeoDataFrame, value_column: str, cmap, norm: Normalize) -> None:
    polygon_geometries = geodataframe.loc[geodataframe.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    line_geometries = geodataframe.loc[geodataframe.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()
    point_geometries = geodataframe.loc[geodataframe.geometry.geom_type.isin(["Point", "MultiPoint"])].copy()

    if len(polygon_geometries) > 0:
        polygon_geometries.plot(ax=axis, column=value_column, cmap=cmap, norm=norm, linewidth=0, alpha=0.95, zorder=2, rasterized=True)
    if len(line_geometries) > 0:
        line_geometries.plot(ax=axis, column=value_column, cmap=cmap, norm=norm, linewidth=0.45, alpha=0.95, zorder=3, rasterized=True)
    if len(point_geometries) > 0:
        point_geometries.plot(ax=axis, column=value_column, cmap=cmap, norm=norm, markersize=4.5, alpha=0.95, zorder=4, rasterized=True)


def prepare_map_axis(axis, title: str, panel_label: str) -> None:
    axis.set_axis_off()
    axis.set_aspect("equal")
    axis.set_title(title, loc="left", pad=2)
    add_panel_label(axis, panel_label)


def plot_coastal_assets(axis, figure, panel_label: str, compact_furniture: bool = False) -> None:
    asset_norm = Normalize(vmin=0, vmax=float(positive_coastal_asset_locations["plot_avoided_ead_usd"].max()))
    prepare_map_axis(axis, "Coastal flood: infrastructure locations", panel_label)
    main_island_boundary.boundary.plot(ax=axis, color=BOUNDARY_COLOR, linewidth=0.65, zorder=5)
    plot_geometry_types(axis, positive_coastal_asset_locations, "plot_avoided_ead_usd", ASSET_CMAP, asset_norm)
    main_island_boundary.boundary.plot(ax=axis, color=BOUNDARY_COLOR, linewidth=0.65, zorder=6)
    set_map_extent(axis, MAIN_ISLAND_BOUNDS)
    add_standard_map_furniture(axis, compact=compact_furniture)
    add_colorbar(figure, axis, ASSET_CMAP, asset_norm, "Avoided EAD (US$)", top_label_prefix=">")


def plot_coastal_mangroves(axis, figure, panel_label: str, compact_furniture: bool = False) -> None:
    mangrove_norm = Normalize(vmin=0, vmax=float(positive_coastal_mangrove_attribution["plot_avoided_ead_thousand_usd"].max()))
    prepare_map_axis(axis, "Coastal flood: mangrove patches providing benefits", panel_label)
    main_island_boundary.boundary.plot(ax=axis, color=BOUNDARY_COLOR, linewidth=0.65, zorder=5)
    coastal_mangrove_attribution.plot(ax=axis, color=NO_BENEFIT_MANGROVE_COLOR, linewidth=0, alpha=1.0, zorder=2)
    positive_coastal_mangrove_attribution.plot(
        ax=axis,
        column="plot_avoided_ead_thousand_usd",
        cmap=MANGROVE_CMAP,
        norm=mangrove_norm,
        edgecolor=POSITIVE_MANGROVE_EDGE_COLOR,
        linewidth=0.18,
        alpha=0.98,
        zorder=3,
    )
    main_island_boundary.boundary.plot(ax=axis, color=BOUNDARY_COLOR, linewidth=0.65, zorder=6)
    set_map_extent(axis, MAIN_ISLAND_BOUNDS)
    add_standard_map_furniture(axis, compact=compact_furniture)
    add_colorbar(figure, axis, MANGROVE_CMAP, mangrove_norm, "Attributed avoided EAD (US$ thousand)", top_label_prefix=">")


def plot_river_avoided_ead(axis, figure, panel_label: str, compact_furniture: bool = False) -> None:
    river_asset_norm = Normalize(vmin=0, vmax=RIVER_AVOIDED_EAD_COLOR_CAP_USD)
    prepare_map_axis(axis, "River flood: infrastructure locations", panel_label)
    axis.imshow(
        river_avoided_ead["data"],
        norm=river_asset_norm,
        cmap=ASSET_CMAP,
        extent=river_avoided_ead["extent"],
        origin="upper",
        interpolation="nearest",
        zorder=2,
    )
    main_island_boundary.boundary.plot(ax=axis, color=BOUNDARY_COLOR, linewidth=0.65, zorder=6)
    set_map_extent(axis, MAIN_ISLAND_BOUNDS)
    add_standard_map_furniture(axis, compact=compact_furniture)
    add_colorbar(figure, axis, ASSET_CMAP, river_asset_norm, "Avoided EAD (US$)", top_label_prefix=">")


def plot_river_restoration(axis, figure, panel_label: str, compact_furniture: bool = False) -> None:
    river_restoration_norm = Normalize(vmin=0, vmax=RIVER_RESTORATION_COLOR_CAP_USD)
    prepare_map_axis(axis, "River flood: forest restoration locations", panel_label)
    axis.imshow(
        river_restoration_benefit["data"],
        norm=river_restoration_norm,
        cmap=RIVER_RESTORATION_CMAP,
        extent=river_restoration_benefit["extent"],
        origin="upper",
        interpolation="nearest",
        zorder=2,
    )
    main_island_boundary.boundary.plot(ax=axis, color=BOUNDARY_COLOR, linewidth=0.65, zorder=6)
    set_map_extent(axis, MAIN_ISLAND_BOUNDS)
    add_standard_map_furniture(axis, compact=compact_furniture)
    add_colorbar(figure, axis, RIVER_RESTORATION_CMAP, river_restoration_norm, "Attributed avoided damages (US$)", top_label_prefix=">")


def plot_landslide_assets(
    axis,
    figure,
    panel_label: str,
    geodataframe: gpd.GeoDataFrame,
    title: str,
    compact_furniture: bool = False,
) -> None:
    landslide_asset_norm = Normalize(vmin=0, vmax=float(geodataframe["plot_avoided_ead_usd"].max()))
    prepare_map_axis(axis, title, panel_label)
    main_island_boundary.boundary.plot(ax=axis, color=BOUNDARY_COLOR, linewidth=0.65, zorder=5)
    plot_geometry_types(axis, geodataframe, "plot_avoided_ead_usd", ASSET_CMAP, landslide_asset_norm)
    main_island_boundary.boundary.plot(ax=axis, color=BOUNDARY_COLOR, linewidth=0.65, zorder=6)
    set_map_extent(axis, MAIN_ISLAND_BOUNDS)
    add_standard_map_furniture(axis, compact=compact_furniture)
    add_colorbar(figure, axis, ASSET_CMAP, landslide_asset_norm, "Avoided EAD (US$)", top_label_prefix=">")


def plot_landslide_patch_raster(
    axis,
    figure,
    panel_label: str,
    raster_layer: dict,
    color_cap_usd: float,
    title: str,
    compact_furniture: bool = False,
) -> None:
    patch_norm = Normalize(vmin=0, vmax=float(color_cap_usd))
    prepare_map_axis(axis, title, panel_label)
    axis.imshow(
        raster_layer["data"],
        norm=patch_norm,
        cmap=LANDSLIDE_PATCH_CMAP,
        extent=raster_layer["extent"],
        origin="upper",
        interpolation="nearest",
        zorder=2,
    )
    main_island_boundary.boundary.plot(ax=axis, color=BOUNDARY_COLOR, linewidth=0.65, zorder=6)
    set_map_extent(axis, MAIN_ISLAND_BOUNDS)
    add_standard_map_furniture(axis, compact=compact_furniture)
    add_colorbar(figure, axis, LANDSLIDE_PATCH_CMAP, patch_norm, "Attributed avoided EAD (US$ per cell)", top_label_prefix=">")

def plot_placeholder_panel(axis, title: str, panel_label: str, placeholder_text: str) -> None:
    prepare_map_axis(axis, title, panel_label)
    main_island_boundary.boundary.plot(ax=axis, color="#cfcfcf", linewidth=0.65, zorder=1)
    set_map_extent(axis, MAIN_ISLAND_BOUNDS)
    axis.text(
        0.50,
        0.50,
        placeholder_text,
        transform=axis.transAxes,
        ha="center",
        va="center",
        fontsize=7,
        color="#666666",
    )


def save_figure(figure, output_stem: str, dpi: int = 600) -> None:
    for extension in ["png", "pdf", "svg"]:
        figure.savefig(OUTPUT_DIR / f"{output_stem}.{extension}", dpi=dpi, bbox_inches="tight", facecolor="white")


## Four-panel comparison

In [ ]:
with mpl.rc_context(NATURE_RC):
    figure, axes = plt.subplots(2, 2, figsize=(9.8, 5.8), constrained_layout=False)
    plt.subplots_adjust(left=0.03, right=0.95, top=0.96, bottom=0.05, wspace=0.18, hspace=0.12)

    plot_coastal_assets(axes[0, 0], figure, "(a)", compact_furniture=True)
    plot_coastal_mangroves(axes[0, 1], figure, "(b)", compact_furniture=True)
    plot_river_avoided_ead(axes[1, 0], figure, "(c)", compact_furniture=True)
    plot_river_restoration(axes[1, 1], figure, "(d)", compact_furniture=True)

    save_figure(figure, "coastal_river_avoided_ead_nbs_locations_comparison_2x2")
    plt.show()


## A4 landscape comparison export

In [ ]:
with mpl.rc_context(NATURE_RC):
    figure, axes = plt.subplots(2, 2, figsize=(11.2, 6.7), constrained_layout=False)
    plt.subplots_adjust(left=0.025, right=0.965, top=0.965, bottom=0.045, wspace=0.10, hspace=0.12)

    plot_coastal_assets(axes[0, 0], figure, "(a)", compact_furniture=True)
    plot_coastal_mangroves(axes[0, 1], figure, "(b)", compact_furniture=True)
    plot_river_avoided_ead(axes[1, 0], figure, "(c)", compact_furniture=True)
    plot_river_restoration(axes[1, 1], figure, "(d)", compact_furniture=True)

    save_figure(figure, "coastal_river_avoided_ead_nbs_locations_comparison_2x2_a4_landscape")
    plt.show()


## Multi-hazard comparison with landslide infrastructure panels

In [ ]:
with mpl.rc_context(NATURE_RC):
    figure, axes = plt.subplots(4, 2, figsize=(11.2, 11.6), constrained_layout=False)
    plt.subplots_adjust(left=0.025, right=0.965, top=0.975, bottom=0.035, wspace=0.10, hspace=0.16)

    plot_coastal_assets(axes[0, 0], figure, "(a)", compact_furniture=True)
    plot_coastal_mangroves(axes[0, 1], figure, "(b)", compact_furniture=True)
    plot_river_avoided_ead(axes[1, 0], figure, "(c)", compact_furniture=True)
    plot_river_restoration(axes[1, 1], figure, "(d)", compact_furniture=True)
    plot_landslide_assets(
        axes[2, 0],
        figure,
        "(e)",
        positive_landslide_protection_locations,
        "Landslide: infrastructure locations from forest protection",
        compact_furniture=True,
    )
    plot_landslide_patch_raster(
        axes[2, 1],
        figure,
        "(f)",
        landslide_protection_patch_benefit,
        LANDSLIDE_PROTECTION_PATCH_COLOR_CAP_USD,
        "Landslide: existing forest patches providing protection benefits",
        compact_furniture=True,
    )
    plot_landslide_assets(
        axes[3, 0],
        figure,
        "(g)",
        positive_landslide_restoration_locations,
        "Landslide: infrastructure locations from forest restoration",
        compact_furniture=True,
    )
    plot_landslide_patch_raster(
        axes[3, 1],
        figure,
        "(h)",
        landslide_restoration_patch_benefit,
        LANDSLIDE_RESTORATION_PATCH_COLOR_CAP_USD,
        "Landslide: restoration areas providing benefits",
        compact_furniture=True,
    )

    save_figure(figure, "coastal_river_landslide_avoided_ead_nbs_locations_comparison_4x2")
    plt.show()


## Stacked comparison

In [ ]:
with mpl.rc_context(NATURE_RC):
    figure, axes = plt.subplots(4, 1, figsize=(7.1, 10.8), constrained_layout=False)
    plt.subplots_adjust(left=0.02, right=0.92, top=0.98, bottom=0.03, hspace=0.09)

    plot_coastal_assets(axes[0], figure, "(a)")
    plot_coastal_mangroves(axes[1], figure, "(b)")
    plot_river_avoided_ead(axes[2], figure, "(c)")
    plot_river_restoration(axes[3], figure, "(d)")

    save_figure(figure, "coastal_river_avoided_ead_nbs_locations_comparison_stacked")
    plt.show()
